# Exercise 1

In [ ]:
from enn554.utilities import read_TMY, satellite_image
import matplotlib.pyplot as plt
from enn554.math import sind, cosd, acosd
from enn554.sun import plane_of_array_irradiance
from enn554.paths import data_dir
from numpy import sign, abs
from scipy.optimize import minimize
dd = data_dir()

## Import TMY data

In [ ]:
tmy_brisbane,metadata = read_TMY(dd/'tutorial_3/brisbane_tmy.csv',now_year=2025)
lat,lon = metadata['Latitude'], metadata['Longitude']
ax = tmy_brisbane.plot(x='Datetime',y='GHI')

## Compute POA irradiance


Define parameters

In [ ]:
L,L_st = 360 - lon, 360-150.0
# beta, gamma = 0,180
rho = 0.0

### Sun angles

In [ ]:
doy = tmy_brisbane['Datetime'].dt.day_of_year
hour_of_day = tmy_brisbane['Datetime'].dt.hour + tmy_brisbane['Datetime'].dt.minute/60

delta = 23.45 * sind(360/365 * (doy - 81))
B = 360/364*(doy - 81)
E = 9.87*sind(2*B) - 7.53*cosd(B) - 1.5*sind(B) # minutes
t_solar = hour_of_day + 4*(L_st - L)/60 + E/60
omega = 15*(t_solar - 12)

zenith = acosd(sind(delta)*sind(lat) + cosd(delta)*cosd(lat)*cosd(omega))
azimuth = sign(omega)* abs(acosd( (cosd(zenith)*sind(lat)-sind(delta))/(sind(zenith)*cosd(lat)) )) # south zero, west positive

tmy_brisbane['zenith'] = zenith
tmy_brisbane['azimuth'] = azimuth

### Compute DHI

In [ ]:
GHI,DNI = tmy_brisbane['GHI'], tmy_brisbane['DNI']
DHI = GHI-DNI*cosd(zenith)

### ISM for horizontal panel & annual insolation

In [ ]:
def POA(tmy_brisbane,beta,gamma,rho):
    zenith, azimuth = tmy_brisbane['zenith'], tmy_brisbane['azimuth']
    cos_AOI = cosd(zenith)*cosd(beta) + sind(zenith)*sind(beta)*cosd(azimuth-gamma)
    mask = (cos_AOI > 0)
    # mask = (cos_AOI > -2.0)
    poa = DNI*cos_AOI + DHI*(1+cosd(beta))/2 + GHI*rho*(1-cosd(beta))/2

    return poa*mask,cos_AOI

tmy_brisbane['POA_flat'],caoi = POA(tmy_brisbane,beta=0,gamma=180,rho=rho)
insolation = sum(tmy_brisbane['POA_flat'])/1000 # kWh/m2

ax = tmy_brisbane.plot(x='Datetime',y=['POA_flat'])
ax.set_xlabel('Date/time')
ax.set_ylabel('Plane of array irradiance (W/m2)')
ax.set_title(f'Insolation: {insolation:.2f} kWh/m2')

## Maximize insolation by altering tilt

In [ ]:
obj = lambda x: -sum(POA(tmy_brisbane,x[0],x[1],rho=rho)[0])
res = minimize(obj,x0=[-lat,180],bounds=((0,90),(0,180)))
print(f"Optimal tilt: {res.x[0]:.1f} degrees")
print(f"Optimal azimuth: {res.x[1]:.1f} degrees")
print(f"Optimal insolation: {-res.fun/1000:.1f} kWh/m2")
tmy_brisbane['POA_opt'],_ = POA(tmy_brisbane,beta=res.x[0],gamma=res.x[1],rho=0)

In [ ]:
G_POA = plane_of_array_irradiance(tmy_brisbane['Datetime'],DNI,GHI,lat,L,L_st,25.1,180,ρ=0.2,model="ISM",θ_z_max=90,clip_aoi_greater_than_90=True,ignore_leapday=False)
sum(G_POA)/1000

# Exercise 2

Download image that will be used for SAM shading analysis

In [ ]:
satellite_image(lat,lon,half_width_m=50,save_path=dd/'tutorial_3/brisbane_satellite.png')
fig,ax = plt.subplots()
ax.imshow(plt.imread(dd/'tutorial_3/brisbane_satellite.png'))

Go to SAM for shading analysis. Now analyze insolation with shading.

In [ ]:
import pandas as pd
fd_i = 1-12.11/100.00
fd_ii = 1.0 - 13.14/100.00
fb_i = 1.0 - pd.read_csv(dd/'tutorial_3/shade_60min_i.csv',skiprows=4).iloc[:,-1].values/100.00
fb_ii = 1.0 - pd.read_csv(dd/'tutorial_3/shade_60min_ii.csv',skiprows=4).iloc[:,-1].values/100.00

In [ ]:
def POA_with_blocking(tmy_brisbane,beta,gamma,fb,fd):
    zenith, azimuth = tmy_brisbane['zenith'], tmy_brisbane['azimuth']
    cos_AOI = cosd(zenith)*cosd(beta) + sind(zenith)*sind(beta)*cosd(azimuth-gamma)
    mask = (cos_AOI > 0)
    # mask = (cos_AOI > -2.0)
    poa = fb*DNI*cos_AOI + fd*DHI*(1+cosd(beta))/2

    return poa*mask,cos_AOI

tmy_brisbane['fb_flat'] = fb_i
tmy_brisbane['POA_blocking_flat'],_ = POA_with_blocking(tmy_brisbane,0,180,fb_i,fd_i)
tmy_brisbane['POA_blocking_opt'],_ = POA_with_blocking(tmy_brisbane,0,180,fb_ii,fd_ii)

In [ ]:
ax = tmy_brisbane[['Datetime','POA_flat','POA_blocking_flat']].plot(x='Datetime')
ax.set_xlabel('Date/time')
ax.set_ylabel('Plane of array irradiance (W/m2)')

In [ ]:
insolation = sum(tmy_brisbane['POA_blocking_flat'])/1000 # kWh/m2
print(f'Insolation on flat panel considering blocking: {insolation:.1f} kWh/m2')

In [ ]:
insolation = tmy_brisbane['POA_blocking_opt'].sum()/1000.00
print(f'Insolation on "optimally" tilted panel considering blocking: {insolation:.1f} kWh/m2')